# 03 - A SpanProcessor for per-request context

Goal: stamp values that change per request (channel, user, conversation, caller) onto every span, before export.

A resource is fixed for the process. A customer's **channel** and **user** differ for every turn. OpenTelemetry carries per-request values in **baggage**, and a `SpanProcessor` sees every span as it starts,
before the exporter. So:

1. The request handler puts values in baggage (`bank_agent.ask(..., baggage_values=...)` does this).
2. Your processor reads the baggage in `on_start` and sets `langsmith.metadata.<key>` attributes on the span.

One wrinkle: LangChain creates its spans internally, and baggage may not be visible when it does. So the processor reads the baggage at the **first** span of a trace (the agent's root span, where it is visible), remembers it for that trace id,
and stamps every later span in the same trace.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

## Step 1: Write the processor

Fill in `on_start(self, span, parent_context=None)`:

1. `bag = baggage.get_all(parent_context or context.get_current())`
2. `trace_id = span.get_span_context().trace_id`
3. If `bag` is not empty, remember it: `self.by_trace[trace_id] = dict(bag)`
4. If `self.by_trace.get(trace_id)` exists, set it on the span with `span.set_attributes(otel_setup.to_langsmith_attributes(meta))`

In [ ]:
# Reference solution: try writing this yourself first.
class RequestContextProcessor(SpanProcessor):
    """Challenge 3: stamp per-request baggage onto every span in the trace."""

    def __init__(self):
        self.by_trace = {}

    def on_start(self, span, parent_context=None):
        bag = baggage.get_all(parent_context or context.get_current())
        trace_id = span.get_span_context().trace_id
        if bag:
            self.by_trace[trace_id] = dict(bag)
        meta = self.by_trace.get(trace_id)
        if meta:
            span.set_attributes(otel_setup.to_langsmith_attributes(meta))

## Step 2: Register it, and send per-request context

`processors=[...]` run in order, before the exporter. `ref.per_request(conversation_id)` is what a real gateway would put in baggage, from the session store: channel, user, language, modality and the caller identity.
The resource from challenge 2 stays.

In [ ]:
print(ref.per_request("example-conversation-id"))
otel_setup.configure(resource_attributes=ref.resource_attributes(), processors=[RequestContextProcessor()])
agent = bank_agent.build_agent()
bank_traffic.run_turns(bank_agent, agent, bank_traffic.QUESTIONS[:6], per_request=ref.per_request)
otel_setup.flush()
n = bank_traffic.wait_for_runs(PROJECT, minimum=12)
print(f"{n} root runs in {PROJECT}")

## Step 3: See it on the traces

In [ ]:
time.sleep(10)
runs = enrich.agent_runs(enrich.project_runs(client, PROJECT, hours=1))
seen = [(r.name, enrich.meta_of(r)) for r in runs if "channel" in enrich.meta_of(r)]
print(f"{len(seen)} of {len(runs)} spans carry a channel")
for name, m in seen[:6]:
    print(name[:28].ljust(30), m.get("channel"), m.get("user_name"), m.get("caller_identity"))
lab_links.project_link(hint="filter on metadata.channel = mobile_app, then group a chart by metadata.channel")

**Result:** most spans in the new traces carry `channel`, `user_name`, `caller_identity` and `conversation_id`. Spans from before this challenge do not: enrichment before export only applies to what is exported afterwards.

If only the root span has them, LangChain's spans are not nested under the agent's root span in your SDK version. Note it and keep going: later notebooks only need the keys on runs.

Next: challenge 4 records risk and compliance decisions from inside the agent's own code.